In [5]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu từ thư mục raw
print("Đang đọc dữ liệu OULAD...")
student_info = pd.read_csv('../data/studentInfo.csv')
student_vle = pd.read_csv('../data/studentVle.csv')

# 2. Xử lý bảng Hành vi (Lấy tổng số lượt click trên hệ thống LMS của mỗi sinh viên)
# Nhóm theo id_student và tính tổng số click
vle_grouped = student_vle.groupby('id_student')['sum_click'].sum().reset_index()

# 3. Nối (Merge) bảng thông tin sinh viên với bảng hành vi
df_merged = pd.merge(student_info, vle_grouped, on='id_student', how='left')

# Xử lý các sinh viên không có lượt click nào (điền số 0)
df_merged['sum_click'] = df_merged['sum_click'].fillna(0)

# 4. Tạo nhãn dự báo (Target Variable: y)
# Pass/Distinction -> 0 (An toàn)
# Fail/Withdrawn -> 1 (Rủi ro)
def create_label(result):
    if result in ['Pass', 'Distinction']:
        return 0
    return 1

df_merged['Risk_Label'] = df_merged['final_result'].apply(create_label)

# 5. Xem kết quả
print("Kích thước bộ dữ liệu sau khi gộp:", df_merged.shape)
display(df_merged[['id_student', 'sum_click', 'final_result', 'Risk_Label']].head())

# Lưu lại file đã gộp để mồi cho XGBoost
df_merged.to_csv('../data/processed/oulad_ready.csv', index=False)

# 1. Đọc dữ liệu bảng điểm
print("Đang xử lý điểm số...")
student_assessment = pd.read_csv('../data/studentAssessment.csv')

# 2. Tính điểm trung bình (Mô phỏng GPA) cho từng sinh viên
# Nhóm theo id_student và tính giá trị trung bình (mean) của cột 'score'
gpa_grouped = student_assessment.groupby('id_student')['score'].mean().reset_index()

# Đổi tên cột cho dễ hiểu
gpa_grouped.rename(columns={'score': 'GPA_Mo_Phong'}, inplace=True)

# 3. Nối (Merge) điểm GPA này vào bảng dữ liệu tổng (df_merged) đã tạo ở bước trước
df_merged = pd.merge(df_merged, gpa_grouped, on='id_student', how='left')

# 4. Xử lý dữ liệu khuyết thiếu (Missing values)
# Những sinh viên chưa từng làm bài test nào sẽ bị NaN (Not a Number)
# Chúng ta sẽ điền mặc định là 0 để mô hình không bị lỗi
df_merged['GPA_Mo_Phong'] = df_merged['GPA_Mo_Phong'].fillna(0)

# Xem thử thành quả cuối cùng
print("Dữ liệu sau khi thêm GPA:")
display(df_merged[['id_student', 'sum_click', 'GPA_Mo_Phong', 'Risk_Label']].head())

# Cập nhật lại file CSV đã qua xử lý
df_merged.to_csv('../data/processed/oulad_ready.csv', index=False)

Đang đọc dữ liệu OULAD...
Kích thước bộ dữ liệu sau khi gộp: (32593, 14)


,id_student,sum_click,final_result,Risk_Label
0,11391,934.0,Pass,0
1,28400,1435.0,Pass,0
2,30268,281.0,Withdrawn,1
3,31604,2158.0,Pass,0
4,32885,1034.0,Pass,0


Đang xử lý điểm số...
Dữ liệu sau khi thêm GPA:


,id_student,sum_click,GPA_Mo_Phong,Risk_Label
0,11391,934.0,82.0,0
1,28400,1435.0,66.4,0
2,30268,281.0,0.0,1
3,31604,2158.0,76.0,0
4,32885,1034.0,54.4,0


In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV

# 0. ĐỌC LẠI DỮ LIỆU TỪ FILE CSV
print("Đang tải dữ liệu đã tiền xử lý...")
df_merged = pd.read_csv('../data/processed/oulad_ready.csv')

# 1. CHỌN LỌC ĐẶC TRƯNG (Feature Selection)
# Chỉ lấy đúng 4 cột cần thiết và nhãn dự báo
features = ['sum_click', 'GPA_Mo_Phong', 'num_of_prev_attempts', 'studied_credits']

X = df_merged[features]
y = df_merged['Risk_Label']

# Xử lý nhanh các giá trị rỗng (nếu còn sót lại) bằng cách điền số 0
X = X.fillna(0)

# 2. CHIA TẬP DỮ LIỆU
# Chia 80% để học (Train) và 20% để thi thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. BUILD VÀ HUẤN LUYỆN MÔ HÌNH XGBOOST
print("Đang huấn luyện mô hình XGBoost...")
model = xgb.XGBClassifier(
    n_estimators=100,      
    max_depth=5,           
    learning_rate=0.1,     
    random_state=42
)

# Cho mô hình học từ tập Train
model.fit(X_train, y_train)
print("Huấn luyện thành công!\n")

# 4. ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST
y_pred = model.predict(X_test)

# In kết quả
print("=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===")
print(f"Độ chính xác tổng thể (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

print("Chi tiết các chỉ số:")
print(classification_report(y_test, y_pred))


print("=== PHƯƠNG PHÁP 1: HẠ NGƯỠNG QUYẾT ĐỊNH XUỐNG 0.4 ===")
# Lấy ra mảng xác suất dự báo (thay vì nhãn 0/1)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Ép ngưỡng mới: Nếu xác suất >= 40% thì đánh dấu là Rủi ro (1)
new_threshold = 0.4
y_pred_threshold_tuned = (y_pred_proba >= new_threshold).astype(int)

print("Chi tiết các chỉ số với ngưỡng 0.4:")
print(classification_report(y_test, y_pred_threshold_tuned))
print("-" * 50)


print("=== PHƯƠNG PHÁP 2: TÌM KIẾM SIÊU THAM SỐ (GRID SEARCH) ===")
print("Hệ thống đang tự động thử nghiệm nhiều bộ thông số. Quá trình này có thể mất 1-2 phút...\n")

# 1. Định nghĩa lưới các thông số cần thử nghiệm
param_grid = {
    'max_depth': [3, 5, 7],           # Độ sâu của cây quyết định
    'learning_rate': [0.01, 0.1, 0.2],# Tốc độ học
    'n_estimators': [100, 200],       # Số lượng cây
    'scale_pos_weight': [1, 1.2]      # Trọng số ưu tiên nhãn Rủi ro (để tăng Recall)
}

# 2. Khởi tạo một mô hình XGBoost trống
xgb_tuner = xgb.XGBClassifier(random_state=42)

# 3. Sử dụng GridSearch để dò tìm (tập trung tối ưu Recall)
grid_search = GridSearchCV(
    estimator=xgb_tuner, 
    param_grid=param_grid, 
    scoring='recall', # Ra lệnh cho máy ưu tiên chỉ số Recall
    cv=3,             # Chia tập Train làm 3 phần để test chéo
    verbose=1         # Hiển thị tiến trình chạy
)

# Cho máy bắt đầu dò tìm
grid_search.fit(X_train, y_train)

# 4. In ra bộ thông số xịn nhất mà máy tìm được
print("\nBộ thông số TỐT NHẤT tìm được:")
print(grid_search.best_params_)

# 5. Dùng mô hình tốt nhất đó để dự báo lại
best_model = grid_search.best_estimator_
y_pred_grid = best_model.predict(X_test)

print("\nChi tiết các chỉ số của Mô hình Tối ưu:")
print(classification_report(y_test, y_pred_grid))

Đang tải dữ liệu đã tiền xử lý...
Đang huấn luyện mô hình XGBoost...
Huấn luyện thành công!

=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===
Độ chính xác tổng thể (Accuracy): 0.7906

Chi tiết các chỉ số:
              precision    recall  f1-score   support

           0       0.76      0.82      0.79      3051
           1       0.83      0.77      0.80      3468

    accuracy                           0.79      6519
   macro avg       0.79      0.79      0.79      6519
weighted avg       0.79      0.79      0.79      6519

=== PHƯƠNG PHÁP 1: HẠ NGƯỠNG QUYẾT ĐỊNH XUỐNG 0.4 ===
Chi tiết các chỉ số với ngưỡng 0.4:
              precision    recall  f1-score   support

           0       0.79      0.72      0.76      3051
           1       0.77      0.83      0.80      3468

    accuracy                           0.78      6519
   macro avg       0.78      0.78      0.78      6519
weighted avg       0.78      0.78      0.78      6519

--------------------------------------------------
=== PHƯƠNG PHÁP 

In [3]:
import os

# 1. Tạo thư mục 'models' để chứa file cho gọn gàng (nếu chưa có)
os.makedirs('../models', exist_ok=True)

# 2. Lưu mô hình tốt nhất dưới định dạng JSON
best_model.save_model('../models/xgboost_model.json')

print("Đã lưu file mô hình thành công tại: ../models/xgboost_model.json")

Đã lưu file mô hình thành công tại: ../models/xgboost_model.json


In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np

# Khởi tạo một mô hình trống
model = xgb.XGBClassifier()

# Nạp "bộ não" đã huấn luyện từ file JSON vào
model.load_model('../models/xgboost_model.json')

thong_tin_sinh_vien = {
    'sum_click': [40000],            # Rất chăm chỉ tương tác
    'GPA_Mo_Phong': [100.0],         # Điểm quá trình cực cao
    'num_of_prev_attempts': [5],    # Học lần đầu
    'studied_credits': [0]         # Khối lượng tín chỉ vừa phải
}

# Chuyển thành DataFrame (định dạng bảng mà XGBoost hiểu được)
input_df = pd.DataFrame(thong_tin_sinh_vien)

nhan_du_bao = model.predict(input_df)[0]

# Lấy xác suất / tỷ lệ phần trăm rủi ro (lấy phần tử số 1 trong mảng kết quả)
ty_le_rui_ro = model.predict_proba(input_df)[0][1] * 100

# 3. IN KẾT QUẢ RA MÀN HÌNH
print("=== KẾT QUẢ CẢNH BÁO HỌC VỤ ===")
print(f"Nhãn hệ thống trả về: {nhan_du_bao}")
print(f"Tỷ lệ rủi ro (Rớt/Bỏ học): {ty_le_rui_ro:.2f} %")

if nhan_du_bao == 1:
    print("=> HỆ THỐNG ĐỎ: Cố vấn học tập cần can thiệp ngay!")
else:
    print("=> HỆ THỐNG XANH: Sinh viên đang học tập ổn định.")

=== KẾT QUẢ CẢNH BÁO HỌC VỤ ===
Nhãn hệ thống trả về: 0
Tỷ lệ rủi ro (Rớt/Bỏ học): 27.42 %
=> HỆ THỐNG XANH: Sinh viên đang học tập ổn định.


In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo 

print("Đang kết nối và tải dữ liệu từ kho UCI (ID: 697)...")

# 1. Fetch dataset từ hệ thống UCI
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697) 

# 2. Tách dữ liệu thành DataFrames
# X chứa 36 biến số (tín chỉ, điểm số, học phí...)
X = predict_students_dropout_and_academic_success.data.features 
# y chứa cột nhãn (Dropout, Enrolled, Graduate)
y = predict_students_dropout_and_academic_success.data.targets 

# 3. Gộp X và y lại thành một DataFrame duy nhất để dễ phân tích (EDA)
df = pd.concat([X, y], axis=1)

print("Tải dữ liệu thành công! ✅")
print("-" * 50)
print(f"Tổng số dòng (Sinh viên): {df.shape[0]}")
print(f"Tổng số cột (Biến số): {df.shape[1]}")
print("-" * 50)

# Hiển thị thử 5 dòng đầu tiên và danh sách các cột
display(df.head())

Đang kết nối và tải dữ liệu từ kho UCI (ID: 697)...
Tải dữ liệu thành công! ✅
--------------------------------------------------
Tổng số dòng (Sinh viên): 4424
Tổng số cột (Biến số): 37
--------------------------------------------------


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate
